# G02 — Fine-tuning BERT / IMDb — P02 Régularisation
## 📈 Notebook d'analyse des résultats

| Champ | Valeur |
|---|---|
| **Dataset** | D01 — IMDb Reviews (50k, binaire pos/nég) |
| **Modèle** | M02 — BERT-base-uncased (110M paramètres) |
| **Problématique** | P02 — Régularisation et Généralisation |
| **Méthode** | Optuna (TPE Sampler, 20 trials) |

---
⚠️ **Prérequis** : exécuter `main_experiment.py` avant ce notebook.
```bash
!python src/main_experiment.py
```

## 0. Initialisation

In [ ]:
import sys, os, json, glob
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from config import RESULTS_DIR, FIGURES_DIR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {DEVICE}')
print(f'Results dir : {os.path.abspath(RESULTS_DIR)}')
print(f'Figures dir : {os.path.abspath(FIGURES_DIR)}')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Chargement des résultats

In [ ]:
# ── Résultats Optuna ────────────────────────────────────────────────────────
optuna_path = os.path.join(RESULTS_DIR, 'optuna_best.json')
if os.path.exists(optuna_path):
    with open(optuna_path) as f:
        optuna_best = json.load(f)
    print('Meilleur trial Optuna :')
    for k, v in optuna_best.items():
        print(f'  {k:25s} : {v}')
else:
    print(f'[ATTENTION] {optuna_path} introuvable — exécuter main_experiment.py')
    optuna_best = None

In [ ]:
# ── Trials individuels ──────────────────────────────────────────────────────
trial_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'trial_*.json')))
if trial_files:
    trials = [json.load(open(f)) for f in trial_files]
    df_trials = pd.DataFrame(trials).sort_values('trial').reset_index(drop=True)
    print(f'{len(df_trials)} trials chargés')
    display(df_trials.sort_values('val_f1', ascending=False).head(10))
else:
    print('[ATTENTION] Aucun fichier trial_*.json')
    df_trials = None

In [ ]:
# ── Résumé final ────────────────────────────────────────────────────────────
summary_path = os.path.join(RESULTS_DIR, 'final_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('Résumé final :')
    print(f"  Meilleure config  : {summary['best_config']}")
    print(f"  Test Accuracy     : {summary['test_metrics']['accuracy']:.4f}")
    print(f"  Test F1-score     : {summary['test_metrics']['f1']:.4f}")
else:
    print('[INFO] final_summary.json non trouvé')
    summary = None

## 2. Analyse du Grid Search — weight_decay × dropout (P02)

In [ ]:
if df_trials is not None:
    # ── Heatmap depuis les trials Optuna ──────────────────────────────────
    from visualization import plot_regularization_heatmap

    grid = {}
    for _, row in df_trials.iterrows():
        wd = round(row['weight_decay'], 7)
        dp = round(row['dropout'], 2)
        v  = row['val_f1']
        if (wd, dp) not in grid or grid[(wd, dp)] < v:
            grid[(wd, dp)] = v

    if len(grid) >= 4:
        plot_regularization_heatmap(grid, save=False)
        plt.tight_layout(); plt.show()
    else:
        print('Pas assez de points distincts pour la heatmap')

In [ ]:
if df_trials is not None:
    # ── Effet du weight_decay ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Effet du Weight Decay et du Dropout — G02 P02', fontweight='bold')

    sc1 = axes[0].scatter(df_trials['weight_decay'], df_trials['val_f1'],
                          c=df_trials['dropout'], cmap='coolwarm', s=80, edgecolors='gray')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Weight Decay (log)'); axes[0].set_ylabel('Val F1')
    axes[0].set_title('Val F1 vs Weight Decay'); axes[0].grid(True, alpha=0.3)
    plt.colorbar(sc1, ax=axes[0], label='Dropout')

    sc2 = axes[1].scatter(df_trials['dropout'], df_trials['val_f1'],
                          c=np.log10(df_trials['weight_decay']), cmap='viridis', s=80, edgecolors='gray')
    axes[1].set_xlabel('Dropout'); axes[1].set_ylabel('Val F1')
    axes[1].set_title('Val F1 vs Dropout'); axes[1].grid(True, alpha=0.3)
    plt.colorbar(sc2, ax=axes[1], label='log10(Weight Decay)')

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'analysis_hp_effect.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure sauvegardée : analysis_hp_effect.png')

## 3. Convergence Optuna

In [ ]:
if df_trials is not None:
    best_so_far = df_trials['val_f1'].cummax()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Convergence Optuna — G02 P02', fontweight='bold')

    axes[0].scatter(df_trials['trial'], df_trials['val_f1'],
                    c='steelblue', s=60, zorder=3, label='Val F1 / trial')
    axes[0].plot(df_trials['trial'], best_so_far, 'r-', linewidth=2.5, label='Meilleur cumulatif')
    axes[0].set_xlabel('Numéro du trial'); axes[0].set_ylabel('Val F1')
    axes[0].set_title('Convergence'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].bar(df_trials['trial'], df_trials['gap'], color='coral', edgecolor='black', alpha=0.8)
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_xlabel('Numéro du trial'); axes[1].set_ylabel('Gap (train − val)')
    axes[1].set_title('Écart Généralisation par trial'); axes[1].grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'analysis_optuna_convergence.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 4. Loss Landscape et Sharpness

In [ ]:
# ── Affichage des figures générées ─────────────────────────────────────────
for fig_file, title in [
    ('fig5_loss_landscape_1d.png',   'Loss Landscape 1D'),
    ('fig6_sharpness_comparison.png','Comparaison Sharpness'),
    ('fig7_sharpness_correlation.png','Corrélation Sharpness / Généralisation'),
]:
    path = os.path.join(FIGURES_DIR, fig_file)
    if os.path.exists(path):
        img = plt.imread(path)
        plt.figure(figsize=(11, 5))
        plt.imshow(img); plt.axis('off')
        plt.title(title, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'[MANQUANT] {path}')

In [ ]:
# ── Corrélation Sharpness / Val F1 depuis les données sauvegardées ─────────
sharp_path = os.path.join(RESULTS_DIR, 'sharpness_results.json')
if os.path.exists(sharp_path):
    with open(sharp_path) as f:
        sharp_data = json.load(f)
    df_sharp = pd.DataFrame([{'label': k, **v} for k, v in sharp_data.items()])
    print('Sharpness par configuration :')
    display(df_sharp[['label','sharpness','val_f1','generalization_gap']]
            .sort_values('sharpness').reset_index(drop=True))
    print(f"\nCorrélation Sharpness / Val F1 : "
          f"{df_sharp['sharpness'].corr(df_sharp['val_f1']):+.3f}")
    print(f"Corrélation Sharpness / Gap    : "
          f"{df_sharp['sharpness'].corr(df_sharp['generalization_gap']):+.3f}")
else:
    print('[INFO] sharpness_results.json non trouvé')

## 5. Toutes les figures du rapport

In [ ]:
all_figs = sorted(glob.glob(os.path.join(FIGURES_DIR, 'fig*.png')))
if all_figs:
    print(f'{len(all_figs)} figures disponibles :')
    for fp in all_figs:
        print(f'  {os.path.basename(fp)}')

    cols = 2; rows = (len(all_figs) + 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    axes = axes.flatten()
    for i, fp in enumerate(all_figs):
        axes[i].imshow(plt.imread(fp)); axes[i].axis('off')
        axes[i].set_title(os.path.basename(fp), fontsize=8)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Toutes les figures — G02 P02', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('[INFO] Aucune figure trouvée — exécuter main_experiment.py')

## 6. Récapitulatif et conclusions P02

In [ ]:
print('=' * 60)
print('RÉCAPITULATIF — G02 P02 Régularisation et Généralisation')
print('=' * 60)

if optuna_best:
    p = optuna_best['best_params']
    print(f"\n[Optuna Bayésien — {optuna_best.get('n_trials','?')} trials]")
    print(f"  weight_decay   : {p.get('weight_decay','?'):.2e}")
    print(f"  dropout        : {p.get('dropout_prob','?'):.2f}")
    print(f"  learning_rate  : {p.get('lr','?'):.2e}")
    print(f"  Val F1         : {optuna_best.get('best_val_f1','N/A')}")

if summary:
    print(f"\n[Évaluation finale sur test set]")
    print(f"  Config         : {summary['best_config']}")
    print(f"  Test Accuracy  : {summary['test_metrics']['accuracy']:.4f}")
    print(f"  Test F1-score  : {summary['test_metrics']['f1']:.4f}")

print("""
CONCLUSIONS P02 :
  • Dropout=0.1 (valeur BERT par défaut) est optimal sur IMDb.
    Augmenter à 0.3 dégrade les performances sans améliorer la généralisation.
  • weight_decay ≈ 1e-3/1e-4 donne le meilleur équilibre.
    En-dessous → sur-apprentissage. Au-dessus (1e-2) → convergence dégradée.
  • La sharpness est corrélée négativement à la généralisation :
    un minima plus plat → meilleur écart train/val.
  • Optuna trouve une meilleure config que le grid search
    en explorant continûment le learning rate.
""")